# Adam-family benchmark: cells run on Colab (T4 only)

Cell **A** Fashion-MNIST × BN CNN · **B** CIFAR-10 × BN CNN · **D** CIFAR-100 × BN CNN · **E** CIFAR-10 × CNN without BN (ablation).
Cell C (CIFAR-10 × ResNet-18) and, in the published results, cells D and E were run on Kaggle T4 with the same `run_cell.py` commands.

Each Colab session: **Runtime → Run all**. The notebook resumes automatically; finished runs are skipped.

- The code lives in `MyDrive/adam-bench` (upload the whole folder to Drive).
- Results are written straight to `MyDrive/adam-bench/results`, so a lost session loses no data.
- Runtime → Change runtime type → **T4 GPU**. On any other GPU the runner stops.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/adam-bench'   # change if stored elsewhere
RESULTS = f'{PROJECT}/results'
WORK = '/content/adam-bench'                     # copy the code to local disk for speed
DATA = '/content/data'

!rsync -a --delete --exclude results --exclude data {PROJECT}/ {WORK}/
%cd {WORK}
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Pin the PyTorch version

On the first run, the runner records the torch version in `results/<cell>/env_lock.json`. In later sessions this cell reinstalls that version
if Colab has upgraded. If it reinstalls: **Runtime → Restart session**, then Run all again.

In [ ]:
import json, os, re, subprocess, sys, torch, torchvision
lock = None
for c in ('A', 'B', 'D', 'E'):
    p = f'{RESULTS}/{c}/env_lock.json'
    if os.path.exists(p):
        lock = json.load(open(p)); break

if lock is None:
    print('No environment lock yet (first run). Using torch', torch.__version__)
elif lock['torch'] == torch.__version__:
    print('torch matches the lock:', torch.__version__)
else:
    want, want_tv = lock['torch'], lock.get('torchvision')
    m = re.search(r'\+(cu\d+)', want)
    idx = f'https://download.pytorch.org/whl/{m.group(1)}' if m else 'https://download.pytorch.org/whl/cpu'
    base = lambda v: v.split('+')[0]
    pkgs = [f'torch=={base(want)}'] + ([f'torchvision=={base(want_tv)}'] if want_tv else [])
    print(f'Colab has torch {torch.__version__}, the lock requires {want}. Reinstalling:', pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, '--index-url', idx])
    print('\n>>> REINSTALLED. Runtime → Restart session, then Run all again. <<<')
    raise SystemExit

## Self-test (first session only, ~5 minutes)

Must end with `SELFTEST PASSED`. After one success, set `RUN_SELFTEST = False`.

`RUN_CHECK_REF = True` (optional): installs the original packages (timm, schedulefree, …) and compares the re-implemented optimizers with the authors' versions.

In [ ]:
RUN_SELFTEST = True    # enable once on every new machine or after adding optimizers
RUN_CHECK_REF = True   # compare the re-implemented optimizers with the original code
if RUN_SELFTEST:
    !python run_cell.py --selftest --out /content/tmp_selftest
if RUN_CHECK_REF:
    !pip install -q timm schedulefree adabelief-pytorch adamp pytorch-optimizer
    !python run_cell.py --check-ref

## Run cells A → B

Each cell: LR sweep (one grid per optimizer, extended automatically if the best LR lies at the edge), then the main runs (5 seeds).
If a session is interrupted, Run all in the next session continues where it stopped. When optimizers are added to `bench/config.py` after a cell has fixed its LRs, the runner sweeps and runs only the new ones.

To measure the real runtime first: `!python run_cell.py --cell A --smoke --data {DATA} --out /content/tmp_smoke`.

In [ ]:
for cell in ['A', 'B']:   # C, D, E were run on Kaggle
    print(f'\n========== Cell {cell} ==========', flush=True)
    p = subprocess.Popen([sys.executable, '-u', 'run_cell.py', '--cell', cell, '--phase', 'all',
                          '--out', RESULTS, '--data', DATA],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if rc != 0:
        print(f'Cell {cell} stopped with code {rc}. See the messages above and {RESULTS}/{cell}/log.txt')
        break

## Progress

In [ ]:
for c in ['A', 'B']:
    !python run_cell.py --cell {c} --phase status --out {RESULTS}